<a href="https://colab.research.google.com/github/Fu-Pei-Yin/Deep-Generative-Mode/blob/week9/LoRA_Zero_shot_Few_shot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
HW9 - LLM 微調：情緒分類與憂鬱症風險監測
包含 Zero-shot、Few-shot、LoRA 微調三種方法
CPU/GPU 通用版本
"""

# ==================== 1. 環境安裝 ====================
!pip install -q transformers datasets peft accelerate
!pip install -q scikit-learn matplotlib seaborn pandas numpy torch

print("套件安裝完成！")

# ==================== 2. 主程式 ====================
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

# 檢查 CUDA 可用性
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n使用裝置: {device}")
if torch.cuda.is_available():
    print(f"GPU 裝置: {torch.cuda.get_device_name(0)}")
else:
    print("注意: 使用 CPU 訓練會較慢，建議啟用 GPU")

# 設定隨機種子
np.random.seed(42)
torch.manual_seed(42)

# ==================== 3. 載入資料集 ====================
print("\n=== 載入 Emotion Dataset ===")
dataset = load_dataset("dair-ai/emotion")

print(f"訓練集: {len(dataset['train'])} 筆")
print(f"驗證集: {len(dataset['validation'])} 筆")
print(f"測試集: {len(dataset['test'])} 筆")

# 情緒標籤對應
emotion_labels = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

# 顯示範例
print("\n資料範例:")
for i in range(3):
    example = dataset['train'][i]
    print(f"文本: {example['text']}")
    print(f"情緒: {emotion_labels[example['label']]}\n")

# ==================== 4. 風險映射規則 ====================
def emotion_to_risk(emotion_label):
    """
    將情緒標籤轉換為風險等級
    joy/love/surprise → 0 = low_risk
    anger/fear → 1 = mid_risk
    sadness → 2 = high_risk
    """
    emotion_name = emotion_labels[emotion_label]

    if emotion_name in ['joy', 'love', 'surprise']:
        return 0  # low_risk
    elif emotion_name in ['anger', 'fear']:
        return 1  # mid_risk
    elif emotion_name == 'sadness':
        return 2  # high_risk
    else:
        return 0

risk_labels = {
    0: "low_risk",
    1: "mid_risk",
    2: "high_risk"
}

# 為資料集添加風險標籤
def add_risk_labels(examples):
    examples['risk'] = [emotion_to_risk(label) for label in examples['label']]
    return examples

dataset = dataset.map(add_risk_labels, batched=True)

print("\n=== 風險映射統計 ===")
train_risks = [emotion_to_risk(label) for label in dataset['train']['label']]
print(f"Low risk: {train_risks.count(0)}")
print(f"Mid risk: {train_risks.count(1)}")
print(f"High risk: {train_risks.count(2)}")

# ==================== 5. 載入模型 ====================
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"\n=== 載入模型: {MODEL_NAME} ===")

# 載入 tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 載入基礎模型
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True
)

if torch.cuda.is_available():
    model = model.to('cuda')

print(f"模型載入完成")

# ==================== 6. Zero-shot 推論 ====================
def create_prompt_emotion(text, is_few_shot=False, examples=None):
    """創建情緒分類的 prompt"""
    if is_few_shot and examples:
        prompt = "Classify the emotion. Options: sadness, joy, love, anger, fear, surprise.\n\n"
        for ex_text, ex_label in examples[:5]:  # 限制示例數量
            prompt += f"Text: {ex_text}\nEmotion: {emotion_labels[ex_label]}\n\n"
        prompt += f"Text: {text}\nEmotion:"
    else:
        prompt = f"Classify the emotion as: sadness, joy, love, anger, fear, or surprise.\n\nText: {text}\nEmotion:"
    return prompt

def predict_emotion(text, model, tokenizer, examples=None):
    """預測情緒"""
    prompt = create_prompt_emotion(text, is_few_shot=(examples is not None), examples=examples)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response[len(prompt):].strip().lower()

    # 解析回應
    for label_id, label_name in emotion_labels.items():
        if label_name in response:
            return label_id
    return 1  # 預設返回 joy

print("\n=== Zero-shot 推論測試 ===")
test_samples = dataset['test'].select(range(200))
zero_shot_predictions = []

for i, sample in enumerate(test_samples):
    if i % 50 == 0:
        print(f"處理進度: {i}/{len(test_samples)}")
    pred = predict_emotion(sample['text'], model, tokenizer)
    zero_shot_predictions.append(pred)

# 計算效能
true_labels = [sample['label'] for sample in test_samples]
zero_shot_risks = [emotion_to_risk(pred) for pred in zero_shot_predictions]
true_risks = [emotion_to_risk(label) for label in true_labels]

print("\nZero-shot 情緒分類結果:")
print(f"F1 Score: {f1_score(true_labels, zero_shot_predictions, average='weighted'):.4f}")
print("\nZero-shot 風險分類結果:")
print(f"F1 Score: {f1_score(true_risks, zero_shot_risks, average='weighted'):.4f}")

# ==================== 7. Few-shot 推論 ====================
few_shot_examples = []
for emotion_id in range(6):
    examples_of_emotion = [ex for ex in dataset['train'] if ex['label'] == emotion_id]
    if examples_of_emotion:
        few_shot_examples.append((examples_of_emotion[0]['text'], examples_of_emotion[0]['label']))

print(f"\n=== Few-shot 推論測試 (使用 {len(few_shot_examples)} 個示例) ===")

few_shot_predictions = []
for i, sample in enumerate(test_samples):
    if i % 50 == 0:
        print(f"處理進度: {i}/{len(test_samples)}")
    pred = predict_emotion(sample['text'], model, tokenizer, examples=few_shot_examples)
    few_shot_predictions.append(pred)

few_shot_risks = [emotion_to_risk(pred) for pred in few_shot_predictions]

print("\nFew-shot 情緒分類結果:")
print(f"F1 Score: {f1_score(true_labels, few_shot_predictions, average='weighted'):.4f}")
print("\nFew-shot 風險分類結果:")
print(f"F1 Score: {f1_score(true_risks, few_shot_risks, average='weighted'):.4f}")

# ==================== 8. LoRA 微調 ====================
print("\n=== 準備 LoRA 微調 ===")

# 重新載入模型
from peft import LoraConfig, get_peft_model, TaskType

model_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True
)

if torch.cuda.is_available():
    model_lora = model_lora.to('cuda')

# LoRA 配置
lora_config = LoraConfig(
    r=8,  # 降低 rank 以加快訓練
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

# 準備訓練資料
def format_instruction(example):
    text = example['text']
    emotion = emotion_labels[example['label']]
    prompt = f"Classify emotion.\n\nText: {text}\nEmotion: {emotion}{tokenizer.eos_token}"
    return {"text": prompt}

# 使用較少資料加快訓練
train_size = 2000 if device == "cpu" else 3000
val_size = 300 if device == "cpu" else 400

train_dataset = dataset['train'].select(range(train_size))
val_dataset = dataset['validation'].select(range(val_size))

train_dataset = train_dataset.map(format_instruction)
val_dataset = val_dataset.map(format_instruction)

# Tokenization
def tokenize_function(examples):
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=256,
        padding=False
    )
    result["labels"] = result["input_ids"]
    return result

train_dataset_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text', 'label', 'risk']
)
val_dataset_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text', 'label', 'risk']
)

# 資料整理器
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# 訓練參數
training_args = TrainingArguments(
    output_dir="./emotion_lora_model",
    num_train_epochs=1 if device == "cpu" else 2,
    per_device_train_batch_size=1 if device == "cpu" else 2,
    per_device_eval_batch_size=1 if device == "cpu" else 2,
    gradient_accumulation_steps=16 if device == "cpu" else 8,
    learning_rate=2e-4,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    warmup_steps=50,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none"
)

# 訓練器
trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=val_dataset_tokenized,
    data_collator=data_collator
)

print("\n開始訓練...")
print(f"訓練資料: {len(train_dataset_tokenized)} 筆")
print(f"驗證資料: {len(val_dataset_tokenized)} 筆")

trainer.train()
print("\n訓練完成！")

# ==================== 9. LoRA 模型評估 ====================
print("\n=== LoRA 微調模型評估 ===")

lora_predictions = []
for i, sample in enumerate(test_samples):
    if i % 50 == 0:
        print(f"處理進度: {i}/{len(test_samples)}")
    pred = predict_emotion(sample['text'], model_lora, tokenizer)
    lora_predictions.append(pred)

lora_risks = [emotion_to_risk(pred) for pred in lora_predictions]

print("\nLoRA 情緒分類結果:")
print(f"F1 Score: {f1_score(true_labels, lora_predictions, average='weighted'):.4f}")
print("\nLoRA 風險分類結果:")
print(f"F1 Score: {f1_score(true_risks, lora_risks, average='weighted'):.4f}")

# ==================== 10. 完整評估指標 ====================
print("\n" + "="*50)
print("完整評估指標比較")
print("="*50)

methods = ['Zero-shot', 'Few-shot', 'LoRA']
all_predictions = [zero_shot_risks, few_shot_risks, lora_risks]

results_df = pd.DataFrame()

for method, predictions in zip(methods, all_predictions):
    print(f"\n{method} 方法:")

    # F1 Score
    f1_macro = f1_score(true_risks, predictions, average='macro')
    f1_weighted = f1_score(true_risks, predictions, average='weighted')

    # AUROC
    n_classes = 3
    try:
        true_risks_bin = label_binarize(true_risks, classes=[0, 1, 2])
        pred_risks_bin = label_binarize(predictions, classes=[0, 1, 2])

        auroc_scores = []
        for i in range(n_classes):
            if len(np.unique(true_risks_bin[:, i])) > 1:
                auroc = roc_auc_score(true_risks_bin[:, i], pred_risks_bin[:, i])
                auroc_scores.append(auroc)

        auroc = np.mean(auroc_scores) if auroc_scores else 0.5
    except:
        auroc = 0.5

    # PR-AUC
    try:
        pr_auc_scores = []
        for i in range(n_classes):
            if len(np.unique(true_risks_bin[:, i])) > 1:
                pr_auc = average_precision_score(true_risks_bin[:, i], pred_risks_bin[:, i])
                pr_auc_scores.append(pr_auc)

        pr_auc = np.mean(pr_auc_scores) if pr_auc_scores else 0.33
    except:
        pr_auc = 0.33

    print(f"F1 Score (Macro): {f1_macro:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")
    print(f"AUROC: {auroc:.4f}")
    print(f"PR-AUC: {pr_auc:.4f}")

    # 儲存結果
    results_df = pd.concat([results_df, pd.DataFrame({
        'Method': [method],
        'F1_Macro': [f1_macro],
        'F1_Weighted': [f1_weighted],
        'AUROC': [auroc],
        'PR_AUC': [pr_auc]
    })], ignore_index=True)

    # Confusion Matrix
    cm = confusion_matrix(true_risks, predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Low', 'Mid', 'High'],
                yticklabels=['Low', 'Mid', 'High'])
    plt.title(f'{method} - Confusion Matrix\n(Risk Classification)',
              fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{method.lower().replace("-", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(true_risks, predictions,
                                target_names=['Low Risk', 'Mid Risk', 'High Risk'],
                                zero_division=0))

# 結果比較表
print("\n=== 評估結果總表 ===")
print(results_df.to_string(index=False))

# 繪製比較圖
plt.figure(figsize=(12, 6))
x = np.arange(len(methods))
width = 0.2

bars1 = plt.bar(x - width*1.5, results_df['F1_Macro'], width,
                label='F1 Macro', alpha=0.8, color='steelblue')
bars2 = plt.bar(x - width*0.5, results_df['F1_Weighted'], width,
                label='F1 Weighted', alpha=0.8, color='lightcoral')
bars3 = plt.bar(x + width*0.5, results_df['AUROC'], width,
                label='AUROC', alpha=0.8, color='mediumseagreen')
bars4 = plt.bar(x + width*1.5, results_df['PR_AUC'], width,
                label='PR-AUC', alpha=0.8, color='gold')

plt.xlabel('Method', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Performance Comparison Across Methods', fontsize=14, fontweight='bold')
plt.xticks(x, methods)
plt.legend()
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('method_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# ==================== 11. 風險監測視覺化 ====================
print("\n=== 風險監測視覺化 ===")

# 使用測試資料進行視覺化
test_full = dataset['test'].select(range(500))
lora_full_predictions = []

print("生成完整預測資料...")
for i, sample in enumerate(test_full):
    if i % 100 == 0:
        print(f"進度: {i}/500")
    pred = predict_emotion(sample['text'], model_lora, tokenizer)
    lora_full_predictions.append(pred)

# 轉換為風險分數
risk_predictions = [emotion_to_risk(pred) for pred in lora_full_predictions]
high_risk_probs = [1.0 if r == 2 else 0.0 for r in risk_predictions]

# 1. 高風險走勢圖
plt.figure(figsize=(14, 6))
plt.plot(range(len(high_risk_probs)), high_risk_probs,
         alpha=0.5, linewidth=0.8, color='steelblue', label='High Risk Indicator')
plt.scatter(range(len(high_risk_probs)), high_risk_probs,
            alpha=0.3, s=15, color='steelblue')

# 移動平均線
window = 20
if len(high_risk_probs) >= window:
    moving_avg = pd.Series(high_risk_probs).rolling(window=window).mean()
    plt.plot(range(len(moving_avg)), moving_avg,
             color='red', linewidth=2.5, label=f'{window}-point Moving Average')

plt.title('High Risk Probability Trend Over Test Samples',
          fontsize=14, fontweight='bold')
plt.xlabel('Sample Index', fontsize=12)
plt.ylabel('P(High Risk)', fontsize=12)
plt.ylim(-0.1, 1.1)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('high_risk_trend.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. 高風險濃度熱圖
print("生成高風險濃度熱圖...")
window_size = 50
stride = 10

heatmap_data = []
for i in range(0, len(high_risk_probs) - window_size, stride):
    window_data = high_risk_probs[i:i+window_size]
    heatmap_data.append(window_data)

if len(heatmap_data) > 0:
    heatmap_array = np.array(heatmap_data)

    plt.figure(figsize=(14, 8))
    sns.heatmap(heatmap_array, cmap='YlOrRd',
                cbar_kws={'label': 'High Risk Probability'},
                xticklabels=10, yticklabels=5)
    plt.title('High Risk Concentration Heatmap (Rolling Window)',
              fontsize=14, fontweight='bold')
    plt.xlabel('Position in Window', fontsize=12)
    plt.ylabel('Window Index', fontsize=12)
    plt.tight_layout()
    plt.savefig('high_risk_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

# 3. 風險分布統計
risk_counts = pd.Series(risk_predictions).value_counts().sort_index()
plt.figure(figsize=(10, 6))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars = plt.bar(['Low Risk', 'Mid Risk', 'High Risk'],
               [risk_counts.get(0, 0), risk_counts.get(1, 0), risk_counts.get(2, 0)],
               color=colors, alpha=0.8, edgecolor='black', linewidth=2)

plt.title('Risk Level Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Count', fontsize=12)
plt.xlabel('Risk Level', fontsize=12)

# 數值標籤
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}\n({int(height)/len(risk_predictions)*100:.1f}%)',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('risk_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# ==================== 12. 結果摘要 ====================
print("\n" + "="*60)
print("結果摘要")
print("="*60)

summary = f"""
【資料集資訊】
- 來源: Emotion Dataset (Saravia et al., 2018)
- HuggingFace: dair-ai/emotion
- 訓練集: {len(dataset['train'])} 筆
- 驗證集: {len(dataset['validation'])} 筆
- 測試集: {len(dataset['test'])} 筆
- 情緒類別: 6 類 (joy, love, surprise, anger, fear, sadness)

【風險映射規則】
- Low Risk (0): joy, love, surprise
- Mid Risk (1): anger, fear
- High Risk (2): sadness

【模型設定】
- 基礎模型: {MODEL_NAME}
- 裝置: {device.upper()}
- 精度: {'FP16' if torch.cuda.is_available() else 'FP32'}
- LoRA Rank: 8
- LoRA Alpha: 16
- Target Modules: q_proj, v_proj
- 訓練資料: {train_size} 筆
- 訓練 Epochs: {1 if device == 'cpu' else 2}

【評估結果】
"""

print(summary)
print(results_df.to_string(index=False))

print("\n" + "="*60)
print("所有分析完成！")
print("="*60)

print("\n產生的檔案:")
files = [
    "confusion_matrix_zero_shot.png",
    "confusion_matrix_few_shot.png",
    "confusion_matrix_lora.png",
    "method_comparison.png",
    "high_risk_trend.png",
    "high_risk_heatmap.png",
    "risk_distribution.png"
]
for f in files:
    print(f"  ✓ {f}")

print("\n建議後續步驟:")
print("  1. 下載所有圖片")
print("  2. 記錄評估結果")
print("  3. 撰寫技術報告")
print("  4. 分析模型限制與潛在偏差")
print("  5. 上傳程式碼到 GitHub")

套件安裝完成！

使用裝置: cpu
注意: 使用 CPU 訓練會較慢，建議啟用 GPU

=== 載入 Emotion Dataset ===
訓練集: 16000 筆
驗證集: 2000 筆
測試集: 2000 筆

資料範例:
文本: i didnt feel humiliated
情緒: sadness

文本: i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
情緒: sadness

文本: im grabbing a minute to post i feel greedy wrong
情緒: anger



Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


=== 風險映射統計 ===
Low risk: 7238
Mid risk: 4096
High risk: 4666

=== 載入模型: TinyLlama/TinyLlama-1.1B-Chat-v1.0 ===
模型載入完成

=== Zero-shot 推論測試 ===
處理進度: 0/200
處理進度: 50/200
處理進度: 100/200
處理進度: 150/200

Zero-shot 情緒分類結果:
F1 Score: 0.4500

Zero-shot 風險分類結果:
F1 Score: 0.5095

=== Few-shot 推論測試 (使用 6 個示例) ===
處理進度: 0/200
處理進度: 50/200
處理進度: 100/200
處理進度: 150/200

Few-shot 情緒分類結果:
F1 Score: 0.0008

Few-shot 風險分類結果:
F1 Score: 0.2335

=== 準備 LoRA 微調 ===
